# 詩織 LoRA テスト生成専用ノートブック（Kaggle）

学習済みLoRA（shiori-sdxl-lora Version 1 の出力）を **Add Input** で接続して使う。
学習は行わず、torchao のバージョン不整合を回避して推論だけを実行する。


In [ ]:
# GPU確認
!nvidia-smi


In [ ]:
# 依存関係の用意（torchao は古い版が入っていると peft が ImportError になるため外す）
!pip install -q diffusers transformers accelerate peft huggingface_hub
!pip uninstall -y torchao

# ベースモデル（Animagine XL 4.0）を取得
from huggingface_hub import hf_hub_download
import shutil, os
p = hf_hub_download(repo_id='cagliostrolab/animagine-xl-4.0', filename='animagine-xl-4.0.safetensors')
shutil.copy(p, '/kaggle/working/animagine-xl-4.0.safetensors')
print('model ready:', os.path.getsize('/kaggle/working/animagine-xl-4.0.safetensors'))


In [ ]:
# LoRA を読み込んでテスト生成
import glob, os, torch
from diffusers import StableDiffusionXLPipeline
cands = [c for c in glob.glob('/kaggle/input/**/shiori_oneesan.safetensors', recursive=True)]
print('lora candidates:', cands)
lora_path = cands[0]
pipe = StableDiffusionXLPipeline.from_single_file('/kaggle/working/animagine-xl-4.0.safetensors', torch_dtype=torch.float16).to('cuda')
pipe.load_lora_weights(os.path.dirname(lora_path), weight_name='shiori_oneesan.safetensors')
prompt = 'shiori_oneesan, 1woman, long black hair, straight hair, bangs, pink eyes, gentle smile, cardigan, shopping street, cafe terrace, sunny day'
neg = 'lowres, bad anatomy, bad hands, extra fingers, worst quality, blurry, multiple girls'
img = pipe(prompt, negative_prompt=neg, width=1024, height=1024, num_inference_steps=28, guidance_scale=7).images[0]
os.makedirs('/kaggle/working/output', exist_ok=True)
img.save('/kaggle/working/output/test_outing.png')
print('saved test_outing.png')
img
